# Eksperimen Baseline & Incremental NLP (Tugas 2)
## Prediksi Pergerakan Nilai Tukar USD/IDR: Market vs Sentimen vs TF-IDF

**Kelompok Ayam Bakar Pak Yanto**  
- Aloysius Pijar Hutama Indrianto (24/534591/PA/22675)
- Danar Fathurahman (24/5348200/PA/22828)
- Gilbert Nathaniel (24/533877/PA/22623)
- Satya Wira Pramudita (24/543649/PA/23102)

---

### Research Question
> *Apakah informasi NLP dari teks berita geopolitik memberikan sinyal prediktif tambahan untuk arah pergerakan nilai tukar USD/IDR keesokan harinya dibandingkan hanya data pasar historis saja?*

### Matriks 4 Model yang Dievaluasi
1. **Model 0 (Market-Only):** Baseline historis kurs murni (ARIMAX & XGBoost).
2. **Model 1 (Market + Sentiment):** Fitur kurs + Sentimen VADER (Judul) & Loughran-McDonald (Isi).
3. **Model 2 (Market + TF-IDF):** Fitur kurs + Vektor TF-IDF unigram/bigram (fit HANYA pada Train).
4. **Model 3 (Full Combined):** Mengintegrasikan Kurs + Sentimen + TF-IDF.

### Metrik Evaluasi & Anti-Leakage Rules
- **Target ($y_t$):** Arah biner esok hari ($1$ = NAIK jika $Close_{t+1} > Close_t$, $0$ = TURUN).
- **Metrik:** **F1-Score (Macro, Weighted, per kelas)** dan **Confusion Matrix**.
- **Split Kronologis:** Train (2021–2024; 790 hari), Validation (2025; 237 hari), Test (2026; 154 hari). Zero data leakage.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pastikan root project terdaftar di sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.feature_market import extract_market_features, get_market_feature_cols
from src.feature_sentiment import extract_daily_sentiment_features
from src.feature_tfidf import build_daily_tfidf_features
from src.data_preparation import split_chronological
from src.evaluate import compute_metrics, print_evaluation, build_comparison_dataframe
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBClassifier

## 1. Memuat Data & Rekayasa Fitur Komprehensif
Memproses 14.941 artikel geopolitik bersih dan menyelaraskannya dengan kurs harian Bank Indonesia.

In [ ]:
NEWS_PATH = project_root / "data/cleaned/news_cleaned.csv"
ALIGN_PATH = project_root / "data/aligned/news_alignment.csv"
MARKET_PATH = project_root / "data/aligned/daily_aligned_dataset.csv"
LM_DICT_PATH = project_root / "data/Loughran-McDonald_MasterDictionary.csv"

# 1. Fitur Market
raw_market = pd.read_csv(MARKET_PATH)
df_market = extract_market_features(raw_market)
market_cols = get_market_feature_cols()

# 2. Fitur Sentimen (VADER Judul + LM Isi)
news_df = pd.read_csv(NEWS_PATH)
align_df = pd.read_csv(ALIGN_PATH)
df_sentiment, sentiment_cols = extract_daily_sentiment_features(news_df, align_df, str(LM_DICT_PATH))

# 3. Fitur TF-IDF (Fit HANYA pada Train)
df_tfidf, tfidf_cols, tfidf_vec = build_daily_tfidf_features(news_df, align_df, train_end_date="2024-12-31", max_features=30)

# 4. Penggabungan
df_all = df_market.merge(df_sentiment, on="trading_date", how="left").merge(df_tfidf, on="trading_date", how="left")
all_cols = market_cols + sentiment_cols + tfidf_cols
df_all[all_cols] = df_all[all_cols].fillna(0.0)

train_df, val_df, test_df = split_chronological(df_all)
print(f"Data Train      : {len(train_df)} hari bursa")
print(f"Data Validation : {len(val_df)} hari bursa")
print(f"Data Test       : {len(test_df)} hari bursa")

## 2. Eksperimen Seluruh Model (Model 0 s/d Model 3)

In [ ]:
eval_results = []

def run_eval(clf, features, model_label):
    clf.fit(train_df[features], train_df["target_up"])
    # Validation
    p_val = clf.predict(val_df[features])
    r_val = compute_metrics(val_df["target_up"], p_val, model_label, "Validation")
    eval_results.append(r_val)
    # Test
    p_test = clf.predict(test_df[features])
    r_test = compute_metrics(test_df["target_up"], p_test, model_label, "Test")
    eval_results.append(r_test)

# --- Model 0: Market-Only (ARIMAX & XGBoost) ---
arimax_exog = ["return_lag1", "volatility_5d", "sma5_ratio", "momentum_5d"]
model_arimax = SARIMAX(
    endog=train_df["next_return"], exog=train_df[arimax_exog],
    order=(1, 0, 1), enforce_stationarity=False, enforce_invertibility=False
).fit(disp=False)

p_val_arimax = (model_arimax.apply(val_df["next_return"], exog=val_df[arimax_exog], refit=False).fittedvalues > 0).astype(int)
eval_results.append(compute_metrics(val_df["target_up"], p_val_arimax, "Model 0 - ARIMAX (Market)", "Validation"))

p_test_arimax = (model_arimax.apply(test_df["next_return"], exog=test_df[arimax_exog], refit=False).fittedvalues > 0).astype(int)
eval_results.append(compute_metrics(test_df["target_up"], p_test_arimax, "Model 0 - ARIMAX (Market)", "Test"))

xgb_m0 = XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.05, random_state=42, eval_metric="logloss")
run_eval(xgb_m0, market_cols, "Model 0 - XGBoost (Market)")

# --- Model 1: Market + Sentiment ---
xgb_m1 = XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.05, random_state=42, eval_metric="logloss")
run_eval(xgb_m1, market_cols + sentiment_cols, "Model 1 - XGBoost (Market + Sentiment)")

# --- Model 2: Market + TF-IDF ---
xgb_m2 = XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.05, random_state=42, eval_metric="logloss")
run_eval(xgb_m2, market_cols + tfidf_cols, "Model 2 - XGBoost (Market + TF-IDF)")

# --- Model 3: Full Combined ---
xgb_m3 = XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.05, random_state=42, eval_metric="logloss")
run_eval(xgb_m3, all_cols, "Model 3 - XGBoost (Full Combined)")

summary_df = build_comparison_dataframe(eval_results)
display(summary_df)

## 3. Visualisasi Confusion Matrix Seluruh Model pada Data Test

In [ ]:
test_results = [r for r in eval_results if r["split"] == "Test"]

fig, axes = plt.subplots(1, len(test_results), figsize=(4 * len(test_results), 3.5))
for i, res in enumerate(test_results):
    ax = axes[i]
    sns.heatmap(res["confusion_matrix"], annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Pred 0", "Pred 1"], yticklabels=["Act 0", "Act 1"])
    ax.set_title(f"{res['model']}\n(F1-Macro: {res['f1_macro']:.3f})", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Visualisasi Pentingnya Fitur TF-IDF (Feature Importance)
Melihat kata/topik geopolitik apa yang paling berpengaruh terhadap pergerakan Dolar AS.

In [ ]:
m2_features = market_cols + tfidf_cols
importances = pd.Series(xgb_m2.feature_importances_, index=m2_features).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(15).plot(kind="barh", color="steelblue").invert_yaxis()
plt.title("Top 15 Feature Importances - Model 2 (Market + TF-IDF)", fontsize=12, fontweight="bold")
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()